# 05 — Beijing ML Forecasting

Baseline RF / Extra Trees / XGBoost. Kết quả lưu trong son_nb/output.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from xgboost import XGBRegressor
    HAS_XGB=True
except Exception as e:
    HAS_XGB=False
    print('XGBoost import failed:',e)
PROJECT_ROOT=Path.cwd().parents[2]
SON_OUTPUT=PROJECT_ROOT/'notebooks'/'son_nb'/'output'
WORK_DATA=SON_OUTPUT/'data'; WORK_FIG=SON_OUTPUT/'figures'; WORK_TABLE=SON_OUTPUT/'tables'; WORK_PRED=SON_OUTPUT/'predictions'; WORK_MODEL=SON_OUTPUT/'models'

In [ ]:
train=pd.read_csv(WORK_DATA/'beijing_train_working.csv',parse_dates=['datetime']); valid=pd.read_csv(WORK_DATA/'beijing_valid_working.csv',parse_dates=['datetime']); test=pd.read_csv(WORK_DATA/'beijing_test_working.csv',parse_dates=['datetime'])
features=[c for c in train.columns if c not in ['datetime','target_pm25_t1']]
Xtr,ytr=train[features],train['target_pm25_t1']; Xv,yv=valid[features],valid['target_pm25_t1']; Xte,yte=test[features],test['target_pm25_t1']

In [ ]:
models={'RandomForest':RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1),'ExtraTrees':ExtraTreesRegressor(n_estimators=300,random_state=42,n_jobs=-1)}
if HAS_XGB: models['XGBoost']=XGBRegressor(n_estimators=500,max_depth=8,learning_rate=.05,subsample=.8,colsample_bytree=.8,objective='reg:squarederror',random_state=42,n_jobs=-1)

In [ ]:
rows=[]; preds={}
for name,model in models.items():
    model.fit(Xtr,ytr)
    pv=model.predict(Xv); pt=model.predict(Xte)
    for split,y,p in [('validation',yv,pv),('test',yte,pt)]:
        rows.append({'model':name,'split':split,'MAE':mean_absolute_error(y,p),'RMSE':mean_squared_error(y,p)**.5,'R2':r2_score(y,p)})
    preds[name]=pd.DataFrame({'datetime':test.datetime,'actual_pm25':yte.to_numpy(),'predicted_pm25':pt})
    imp=pd.DataFrame({'feature':features,'importance':model.feature_importances_}).sort_values('importance',ascending=False)
    imp.to_csv(WORK_TABLE/f'{name}_feature_importance.csv',index=False)
metrics=pd.DataFrame(rows); display(metrics); metrics.to_csv(WORK_TABLE/'beijing_ml_metrics_working.csv',index=False)
for name,p in preds.items(): p.to_csv(WORK_PRED/f'beijing_{name}_test_predictions.csv',index=False)
best=metrics[metrics.split=='validation'].sort_values('RMSE').iloc[0].model; p=preds[best]
fig,ax=plt.subplots(figsize=(14,5)); ax.plot(p.datetime,p.actual_pm25,label='Actual'); ax.plot(p.datetime,p.predicted_pm25,label=f'Predicted — {best}'); ax.legend(); ax.set_title(f'Beijing PM2.5 Forecast — {best}'); fig.tight_layout(); fig.savefig(WORK_FIG/f'beijing_{best}_forecast.png',dpi=200); plt.show()

### Gate 05
Chỉ sau khi baseline ổn định mới tuning, ablation và extreme-event evaluation.